# Survived (Titanic) Life Prediction with Apache Spark

このNotebookでは、Apache Sparkを使用してSurvivedデータセット（タイタニック号の生存者データ）を探索し、生存予測の分類モデルを構築します。

In [ ]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

## 1. 環境設定とライブラリのインポート

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.classification.LogisticRegression
import org.apache.spark.ml.feature.{Imputer, StringIndexer, OneHotEncoder, VectorAssembler}
import org.apache.spark.ml.evaluation.{BinaryClassificationEvaluator, MulticlassClassificationEvaluator}

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("SurvivedExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

## 2. データの読み込み

In [ ]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/Survived.csv")

// カラム名を小文字に変換
val lowercaseDF = df.columns.foldLeft(df) { (currentDF, colName) =>
  currentDF.withColumnRenamed(colName, colName.toLowerCase)
}

println(s"データ件数: ${lowercaseDF.count()}")
println("\nスキーマ:")
lowercaseDF.printSchema()

## 3. データの概要確認

In [ ]:
// 最初の10行を表示
lowercaseDF.show(10, truncate = false)

In [ ]:
// 統計情報
lowercaseDF.describe("age", "sibsp", "parch", "fare").show()

In [ ]:
// 生存者数の確認
lowercaseDF.groupBy("survived").count().show()

In [ ]:
// 性別ごとの生存率
lowercaseDF.groupBy("sex", "survived").count().show()

In [ ]:
// クラスごとの生存率
lowercaseDF.groupBy("pclass", "survived").count().orderBy("pclass").show()

## 4. 欠損値の確認と補完

In [ ]:
// 欠損値の確認
println("欠損値の確認:")
lowercaseDF.select("age", "fare").summary("count").show()

// 欠損値補完（平均値で補完）
val imputer = new Imputer()
  .setInputCols(Array("age", "fare"))
  .setOutputCols(Array("age_imputed", "fare_imputed"))
  .setStrategy("mean")

val imputedDF = imputer.fit(lowercaseDF).transform(lowercaseDF)

println("\n欠損値補完完了")
imputedDF.select("age", "age_imputed", "fare", "fare_imputed").show(5)

## 5. カテゴリカル変数のエンコーディング

In [ ]:
// sex（性別）のエンコーディング
val sexIndexer = new StringIndexer()
  .setInputCol("sex")
  .setOutputCol("sex_index")

val sexEncoder = new OneHotEncoder()
  .setInputCol("sex_index")
  .setOutputCol("sex_vec")

// embarked（乗船港）のエンコーディング
val embarkedIndexer = new StringIndexer()
  .setInputCol("embarked")
  .setOutputCol("embarked_index")
  .setHandleInvalid("keep")

val embarkedEncoder = new OneHotEncoder()
  .setInputCol("embarked_index")
  .setOutputCol("embarked_vec")

val encodePipeline = new Pipeline().setStages(Array(
  sexIndexer, sexEncoder,
  embarkedIndexer, embarkedEncoder
))

val encodedDF = encodePipeline.fit(imputedDF).transform(imputedDF)

println("カテゴリカル変数のエンコーディング完了")
encodedDF.select("sex", "sex_vec", "embarked", "embarked_vec").show(5, truncate = false)

## 6. 外れ値の除去

In [ ]:
// 運賃の外れ値を確認
println("運賃の分布:")
encodedDF.select("fare_imputed").describe().show()

// 異常に高い運賃のデータを除去
val cleanedDF = encodedDF.filter("fare_imputed < 500 AND fare_imputed > 0")

println(s"\n外れ値除去前: ${encodedDF.count()} 件")
println(s"外れ値除去後: ${cleanedDF.count()} 件")
println(s"除去された件数: ${encodedDF.count() - cleanedDF.count()} 件")

## 7. 特徴量の統合

In [ ]:
// 全ての特徴量を1つのベクトルに統合
val assembler = new VectorAssembler()
  .setInputCols(Array(
    "pclass",
    "age_imputed",
    "sibsp",
    "parch",
    "fare_imputed",
    "sex_vec",
    "embarked_vec"
  ))
  .setOutputCol("features")

val assembledDF = assembler.transform(cleanedDF)

println(s"準備後のデータ件数: ${assembledDF.count()}")
assembledDF.select("features", "survived").show(5, truncate = false)

## 8. データの分割

In [ ]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = assembledDF.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

// クラスバランスの確認
println("\n訓練データのクラスバランス:")
trainData.groupBy("survived").count().show()

## 9. Logistic Regressionモデルの訓練

In [ ]:
// ラベルカラムをリネーム
val labeledTrain = trainData.withColumnRenamed("survived", "label")

// Logistic Regressionモデルの作成
val lr = new LogisticRegression()
  .setLabelCol("label")
  .setFeaturesCol("features")
  .setMaxIter(100)
  .setRegParam(0.01)

val pipeline = new Pipeline().setStages(Array(lr))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(labeledTrain)
println("訓練完了！")

## 10. モデルの評価

In [ ]:
// テストデータで予測
val labeledTest = testData.withColumnRenamed("survived", "label")
val predictions = model.transform(labeledTest)

// Accuracy（精度）
val correct = predictions.filter("prediction = label").count()
val total = predictions.count()
val accuracy = correct.toDouble / total

println(f"Accuracy: ${accuracy * 100}%.2f%%")

// AUC（Area Under ROC Curve）
val aucEvaluator = new BinaryClassificationEvaluator()
  .setLabelCol("label")
  .setRawPredictionCol("rawPrediction")
  .setMetricName("areaUnderROC")

val auc = aucEvaluator.evaluate(predictions)
println(f"AUC: ${auc * 100}%.2f%%")

// Precision（適合率）
val truePositives = predictions.filter("prediction = 1 AND label = 1").count().toDouble
val predictedPositives = predictions.filter("prediction = 1").count().toDouble
val precision = if (predictedPositives > 0) truePositives / predictedPositives else 0.0
println(f"Precision: ${precision * 100}%.2f%%")

// Recall（再現率）
val actualPositives = predictions.filter("label = 1").count().toDouble
val recall = if (actualPositives > 0) truePositives / actualPositives else 0.0
println(f"Recall: ${recall * 100}%.2f%%")

// F1 Score
val f1 = if (precision + recall > 0) 2 * (precision * recall) / (precision + recall) else 0.0
println(f"F1 Score: ${f1 * 100}%.2f%%")

## 11. 予測結果の確認

In [ ]:
// 予測結果のサンプル表示
predictions.select(
  "pclass", "sex", "age_imputed", "fare_imputed",
  "label", "prediction", "probability"
).show(15, truncate = false)

In [ ]:
// 混同行列（Confusion Matrix）
println("混同行列:")
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

In [ ]:
// 誤分類されたケースの確認
println("誤分類されたケース（最初の10件）:")
predictions.filter("prediction != label")
  .select("pclass", "sex", "age_imputed", "sibsp", "parch", "fare_imputed", "label", "prediction")
  .show(10, truncate = false)

## 12. モデルの係数確認

In [ ]:
// Logistic Regressionモデルの係数を表示
val lrModel = model.stages(0).asInstanceOf[org.apache.spark.ml.classification.LogisticRegressionModel]

println("モデルの係数:")
println(s"Intercept: ${lrModel.intercept}")
println(s"Coefficients: ${lrModel.coefficients}")
println()

// 特徴量の重要度（係数の絶対値）
val featureNames = Array(
  "pclass", "age_imputed", "sibsp", "parch", "fare_imputed",
  "sex_vec_0", "embarked_vec_0", "embarked_vec_1", "embarked_vec_2"
)

val coefficients = lrModel.coefficients.toArray

println("特徴量の重要度（係数の絶対値）:")
featureNames.zip(coefficients).sortBy(-_._2.abs).foreach { case (name, coef) =>
  println(f"$name%-20s: $coef%.4f")
}

## 13. 生存予測のシミュレーション

In [ ]:
// 性別ごとの生存予測傾向
println("性別ごとの生存予測:")
predictions.groupBy("sex", "prediction").count().orderBy("sex", "prediction").show()

// クラスごとの生存予測傾向
println("\nクラスごとの生存予測:")
predictions.groupBy("pclass", "prediction").count().orderBy("pclass", "prediction").show()

## 14. クリーンアップ

In [ ]:
// SparkSessionの停止
// spark.stop()
println("完了！")